# So sánh mô hình ReINTEL và ViFN


**Notebook 03 — Comparative Analysis**

Notebook so sánh hai model TF-IDF + LinearSVC trong bốn hướng:

- ReINTEL → ReINTEL;
- ReINTEL → ViFN;
- ViFN → ReINTEL;
- ViFN → ViFN.

In-domain cho biết hiệu quả trên dataset cùng nguồn. Cross-domain kiểm tra khả năng chuyển sang dataset khác mà không retrain. Nguồn kết quả: `reports/cross_dataset/cross_dataset_results.csv`.


In [ ]:
from pathlib import Path
import unicodedata

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data").is_dir()
        and (path / "reports").is_dir()
        and (path / "models").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy project root.")

REPORT_PATH = ROOT / "reports/cross_dataset/cross_dataset_results.csv"
REINTEL_MODEL_PATH = ROOT / "models/reintel_v2_reviewed/reintel_final.joblib"
VIFN_MODEL_PATH = ROOT / "models/vifn_v1/tuned_word_bigram.joblib"

def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split()).strip()

pd.set_option("display.float_format", lambda value: f"{value:.4f}")


## 1. Mục tiêu và câu hỏi phân tích


Mục tiêu không phải chọn lại model, mà đánh giá mức phụ thuộc miền dữ liệu.

- **CQ1:** Hai model đạt kết quả in-domain như thế nào?
- **CQ2:** Macro F1 giảm bao nhiêu khi chuyển miền?
- **CQ3:** Precision/Recall lớp 1 thay đổi theo hướng chuyển nào?
- **CQ4:** Kết quả nói gì về source bias, label semantics và dataset shift?

Script duy nhất: [`01_evaluate_cross_dataset.py`](../scripts/03_comparative_analysis/01_evaluate_cross_dataset.py).


## 2. Thiết kế thí nghiệm


Hai model đều dùng word TF-IDF unigram–bigram và LinearSVC balanced, nhưng được fit độc lập:

| Model | Dữ liệu Train | `C` | Vocabulary |
|---|---|---:|---:|
| ReINTEL | 3.663 Development samples | 0,1 | đọc từ model |
| ViFN | 924 Clean Train samples | 1,0 | đọc từ model |

Hai target là ReINTEL Final Test 611 mẫu và ViFN Official Test 282 mẫu. Không model nào được retrain khi chuyển miền.


In [ ]:
models = {
    "ReINTEL": joblib.load(REINTEL_MODEL_PATH),
    "ViFN": joblib.load(VIFN_MODEL_PATH),
}

reintel_test = pd.read_csv(
    ROOT / "data/processed/reintel_v2_reviewed/test.csv"
)
vifn_test = pd.read_csv(ROOT / "data/raw/vifn_v1/test_data.csv")
vifn_test["text"] = vifn_test["content"].map(normalize_text)

datasets = {"ReINTEL": reintel_test, "ViFN": vifn_test}

model_summary = pd.DataFrame([
    {
        "model": name,
        "C": model.named_steps["svm"].C,
        "class_weight": model.named_steps["svm"].class_weight,
        "ngram_range": model.named_steps["tfidf"].ngram_range,
        "vocabulary_size": len(model.named_steps["tfidf"].vocabulary_),
    }
    for name, model in models.items()
])
dataset_summary = pd.DataFrame([
    {
        "dataset": name,
        "samples": len(frame),
        "class_0": int((frame["label"] == 0).sum()),
        "class_1": int((frame["label"] == 1).sum()),
    }
    for name, frame in datasets.items()
])
display(model_summary)
display(dataset_summary)

assert dataset_summary["samples"].tolist() == [611, 282]


## 3. Tái tính và xác minh kết quả


Cell dưới đây chạy bốn phép dự đoán trực tiếp từ hai model đóng băng, tính lại metrics và đối chiếu với CSV. Nó không ghi file và không thay đổi model.

Trong cả hai dataset, nhãn 1 là lớp cần phát hiện (`Unreliable` ở ReINTEL, `Fake` ở ViFN), nhưng cách xây dựng nhãn và nguồn dữ liệu khác nhau. Vì vậy cùng số nhãn không đồng nghĩa hai khái niệm hoàn toàn tương đương.


In [ ]:
recorded = pd.read_csv(REPORT_PATH)
rows = []
confusion_matrices = {}

for source_name, model in models.items():
    for target_name, frame in datasets.items():
        y_true = frame["label"].to_numpy()
        y_pred = model.predict(frame["text"].fillna(""))
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        confusion_matrices[(source_name, target_name)] = np.array([[tn, fp], [fn, tp]])
        rows.append({
            "source_model": source_name,
            "target_dataset": target_name,
            "evaluation_type": "in_domain" if source_name == target_name else "cross_domain",
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "precision_1": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "recall_1": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            "f1_1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        })

recomputed = pd.DataFrame(rows)
key = ["source_model", "target_dataset"]
comparison = recorded.merge(recomputed, on=key, suffixes=("_recorded", "_recomputed"))

metric_columns = ["accuracy", "macro_f1", "precision_1", "recall_1", "f1_1"]
for metric in metric_columns:
    comparison[f"{metric}_diff"] = (
        comparison[f"{metric}_recorded"] - comparison[f"{metric}_recomputed"]
    ).abs()

display(recorded)
assert comparison[[f"{m}_diff" for m in metric_columns]].to_numpy().max() < 1e-12
print("Recorded metrics khớp tuyệt đối với kết quả tính lại.")


## 4. Kết quả in-domain


| Model → Dataset | Accuracy | Macro F1 | Precision lớp 1 | Recall lớp 1 | F1 lớp 1 |
|---|---:|---:|---:|---:|---:|
| ReINTEL → ReINTEL | 0,8642 | 0,7708 | 0,5847 | 0,6699 | 0,6244 |
| ViFN → ViFN | **0,8794** | **0,8788** | **0,9194** | **0,8261** | **0,8702** |

ViFN có in-domain score cao hơn. Không thể kết luận model ViFN “tốt hơn” nói chung vì hai Test sets khác độ khó, nguồn, cách gán nhãn và mức duplicate.


In [ ]:
in_domain = recorded[recorded["evaluation_type"] == "in_domain"].copy()
display(in_domain)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ["ReINTEL", "ViFN"]):
    cm = confusion_matrices[(name, name)]
    ax.imshow(cm, cmap="Blues")
    for (row, col), value in np.ndenumerate(cm):
        ax.text(col, row, value, ha="center", va="center")
    ax.set(
        title=f"{name} in-domain",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=[0, 1],
        yticks=[0, 1],
    )
plt.tight_layout()
plt.show()


## 5. Kết quả cross-domain


| Model → Dataset | Accuracy | Macro F1 | Precision lớp 1 | Recall lớp 1 | F1 lớp 1 |
|---|---:|---:|---:|---:|---:|
| ReINTEL → ViFN | 0,6241 | 0,5875 | 0,7667 | **0,3333** | 0,4646 |
| ViFN → ReINTEL | 0,7070 | 0,5944 | **0,2957** | 0,5340 | 0,3806 |

Hai hướng đều giảm mạnh:

- ReINTEL → ViFN có Precision lớp 1 tương đối cao nhưng Recall chỉ 0,3333: bỏ sót phần lớn Fake.
- ViFN → ReINTEL có Recall 0,5340 nhưng Precision chỉ 0,2957: nhiều Reliable bị cảnh báo Unreliable.


In [ ]:
cross_domain = recorded[recorded["evaluation_type"] == "cross_domain"].copy()
display(cross_domain)

for pair, cm in confusion_matrices.items():
    if pair[0] != pair[1]:
        print(f"{pair[0]} -> {pair[1]}:", cm.tolist())

assert confusion_matrices[("ReINTEL", "ViFN")].tolist() == [[130, 14], [92, 46]]
assert confusion_matrices[("ViFN", "ReINTEL")].tolist() == [[377, 131], [48, 55]]


## 6. Mức suy giảm khi chuyển miền


In [ ]:
degradation_rows = []
for model_name in ["ReINTEL", "ViFN"]:
    own = recorded[
        (recorded["source_model"] == model_name)
        & (recorded["target_dataset"] == model_name)
    ].iloc[0]
    other = recorded[
        (recorded["source_model"] == model_name)
        & (recorded["target_dataset"] != model_name)
    ].iloc[0]
    absolute_drop = own["macro_f1"] - other["macro_f1"]
    degradation_rows.append({
        "model": model_name,
        "in_domain_macro_f1": own["macro_f1"],
        "cross_domain_macro_f1": other["macro_f1"],
        "absolute_drop": absolute_drop,
        "relative_drop": absolute_drop / own["macro_f1"],
    })

degradation = pd.DataFrame(degradation_rows)
display(degradation)


- ReINTEL model giảm Macro F1 khoảng **0,1832**, tương đương **23,8%** so với in-domain.
- ViFN model giảm khoảng **0,2844**, tương đương **32,4%**.

Mặc dù ViFN có in-domain score cao hơn, mức giảm tương đối khi sang ReINTEL lớn hơn. Điều này phù hợp giả thuyết model học tín hiệu gắn với nguồn/chủ đề của ViFN.


In [ ]:
pivot = recorded.pivot(
    index="source_model", columns="target_dataset", values="macro_f1"
)
ax = pivot.plot(kind="bar", figsize=(8, 5), ylim=(0, 1), rot=0)
ax.set_title("Macro F1: in-domain và cross-domain")
ax.set_ylabel("Macro F1")
ax.legend(title="Target dataset")
plt.tight_layout()
plt.show()


## 7. Vì sao chuyển miền khó?


Các nguyên nhân có thể đồng thời tồn tại:

1. **Dataset shift:** chủ đề, thời điểm, độ dài và phong cách khác nhau.
2. **Source bias:** tên nguồn hoặc dấu hiệu xuất bản tương quan mạnh với nhãn trong Train.
3. **Label semantics:** `Unreliable` và `Fake` gần nhau nhưng quy trình gán nhãn có thể khác.
4. **Vocabulary shift:** n-gram quan trọng ở dataset nguồn có thể hiếm hoặc mang nghĩa khác ở target.
5. **Class prior:** tỷ lệ lớp 1 của ReINTEL Test khoảng 16,9%, còn ViFN Test khoảng 48,9%.
6. **Duplicate policy:** ReINTEL dùng group-aware split; ViFN giữ Official Test nguyên vẹn và báo cáo unique-only như phân tích phụ.

Kết quả chỉ cho thấy sự suy giảm kết hợp của các yếu tố này; không cô lập quan hệ nhân quả.


## 8. Discussion và kết luận


- Cả hai model hoạt động tốt nhất trên miền đã huấn luyện.
- Cross-domain Macro F1 chỉ khoảng **0,59**, thấp hơn rõ rệt in-domain.
- Precision/Recall thay đổi bất đối xứng cho thấy cùng threshold quyết định không chuyển ổn định giữa hai dataset.
- Score in-domain cao không đủ để khẳng định khả năng fact-check tổng quát.
- Không nên triển khai model trên nguồn mới mà không có validation target-domain, error analysis và giám sát drift.

Hướng tiếp theo: xây tập đánh giá đa nguồn, source-stratified split, domain adaptation, calibration trên validation target-domain và kết hợp retrieval/evidence thay vì chỉ dựa vào văn bản.


## Phụ lục A — Script và artifact


| Thành phần | Đường dẫn |
|---|---|
| Script so sánh | [`01_evaluate_cross_dataset.py`](../scripts/03_comparative_analysis/01_evaluate_cross_dataset.py) |
| ReINTEL notebook | [`01_reintel_experiments.ipynb`](01_reintel_experiments.ipynb) |
| ViFN notebook | [`02_vifn_experiments.ipynb`](02_vifn_experiments.ipynb) |
| Kết quả so sánh | `reports/cross_dataset/cross_dataset_results.csv` |
| ReINTEL model | `models/reintel_v2_reviewed/reintel_final.joblib` |
| ViFN model | `models/vifn_v1/tuned_word_bigram.joblib` |

~~~bash
python scripts/03_comparative_analysis/01_evaluate_cross_dataset.py
~~~


## Phụ lục B — Kiểm tra đồng bộ


In [ ]:
checks = {
    "four evaluation directions": len(recorded) == 4,
    "two in-domain rows": (recorded["evaluation_type"] == "in_domain").sum() == 2,
    "two cross-domain rows": (recorded["evaluation_type"] == "cross_domain").sum() == 2,
    "artifact equals recomputation": (
        comparison[[f"{m}_diff" for m in metric_columns]].to_numpy().max() < 1e-12
    ),
    "ReINTEL in-domain Macro F1": np.isclose(
        recorded.query(
            'source_model == "ReINTEL" and target_dataset == "ReINTEL"'
        )["macro_f1"].iloc[0],
        0.7707586531115943,
    ),
    "ViFN in-domain Macro F1": np.isclose(
        recorded.query(
            'source_model == "ViFN" and target_dataset == "ViFN"'
        )["macro_f1"].iloc[0],
        0.8788231130883171,
    ),
    "all cross-domain Macro F1 below 0.60": (
        cross_domain["macro_f1"] < 0.60
    ).all(),
}
check_table = pd.Series(checks, name="passed").to_frame()
display(check_table)
assert check_table["passed"].all()
print("OK — comparative notebook đồng bộ với hai frozen models và report.")
